# Uncertainty Quantification for GFlowNet Policies via Polynomial Chaos Expansion

**Interactive demonstration of the PCE surrogate framework**

This notebook walks through the core ideas of the paper in a hands-on way,
using the 5x5 gridworld as a concrete example. It is designed for readers
who may not be familiar with either GFlowNets or polynomial chaos expansions.

---

## The problem in one sentence

> When the reward function that trains a sequential decision-maker is uncertain,
> **which aspects of that uncertainty matter most for which decisions?**

## 1. What is a GFlowNet?

A **Generative Flow Network (GFlowNet)** is a policy that generates structured
objects (molecules, sequences, graphs) by making a series of construction
decisions -- one step at a time -- such that the probability of generating an
object is proportional to its reward.

Think of it as a **stochastic search procedure** that has learned to sample
diverse, high-reward solutions rather than collapsing to a single optimum.

### Why not just optimise?

| Approach | What it finds | Diversity |
|----------|--------------|----------|
| Greedy / RL | Single best solution | None |
| MCMC | Samples from reward distribution | Yes, but slow mixing |
| **GFlowNet** | **Samples proportional to reward** | **Yes, by construction** |

In drug discovery, materials science, or experimental design, we rarely want
just one answer -- we want a diverse set of good candidates. GFlowNets are
designed for exactly this.

### Sequential construction

A GFlowNet builds objects step-by-step:

```
Step 0: Choose first component    -> policy pi(a | state_0)
Step 1: Choose second component   -> policy pi(a | state_1)
  ...                                   ...
Step T: Terminate                  -> final object x, reward R(x)
```

At each step, the policy is a **probability distribution over actions**.
These are the distributions we want to understand the uncertainty of.

### The training objective: trajectory balance

The GFlowNet is trained so that for every complete trajectory $\tau = (s_0, a_0, s_1, a_1, \ldots, s_T)$:

$$Z \cdot \prod_{t} P_F(a_t | s_t) = R(x) \cdot \prod_{t} P_B(s_t | s_{t+1})$$

where $Z$ is a learned partition function and $P_B$ is a backward policy.
In our gridworld, we use the simplified **trajectory balance loss**:

$$\mathcal{L} = \left(\log Z + \sum_t \log P_F(a_t|s_t) - \log R(x)\right)^2$$

## 2. What is a Polynomial Chaos Expansion (PCE)?

A PCE is a **polynomial surrogate** that maps uncertain input parameters to
model outputs, using a special basis that makes variance decomposition trivial.

### The key idea

Suppose a model output $f(\mu)$ depends on uncertain parameters $\mu = (\mu_1, \mu_2)$.
A PCE approximates $f$ as:

$$f(\mu) \approx \sum_{j} c_j \, \psi_j(\mu)$$

where $\psi_j$ are **orthonormal polynomial basis functions** chosen to match
the distribution of $\mu$.

### Why orthonormality matters

If the basis functions satisfy $\mathbb{E}[\psi_i \psi_j] = \delta_{ij}$, then:

$$\text{Var}[f] = \sum_{j \neq 0} c_j^2$$

The total variance **decomposes exactly** into contributions from each coefficient.
This is not an approximation -- it is an algebraic identity.

### Sobol indices fall out for free

A **Sobol sensitivity index** $S_i$ answers: *"What fraction of the output
variance is caused by uncertainty in input $\mu_i$?"*

$$S_i = \frac{\sum_{j: \text{only } \mu_i \text{ active}} c_j^2}{\sum_{j \neq 0} c_j^2}$$

We simply group the squared coefficients by which input dimensions they
involve. **No Monte Carlo sampling needed** -- this is a closed-form
computation from the fitted coefficients.

### Basis choice: Hermite polynomials

| Input distribution | Optimal basis | Why |
|-------------------|--------------|-----|
| Gaussian $\mathcal{N}(0,1)$ | **Hermite** | Orthonormal w.r.t. Gaussian measure |
| Uniform $[-1,1]$ | Legendre | Orthonormal w.r.t. uniform measure |

We use PCA for dimensionality reduction, which produces components that are
approximately Gaussian -- so **Hermite polynomials are the natural match**.

### Embedding choice: PCA vs nonlinear alternatives

The analytical Sobol formula above requires **statistically independent** and
**Gaussian** inputs for exact validity with a Hermite basis:

- **Linear PCA** (our default): uncorrelated by construction, approximately
  Gaussian. Sobol indices are approximate but calibration is excellent.
  Simple, interpretable, and requires no hyperparameter tuning.
- **β-VAE** ($\beta \geq 4$): a nonlinear encoder that can achieve higher
  reconstruction fidelity ($R^2 > 0.99$) while producing approximately
  independent *and* Gaussian latents -- making the analytical Sobol
  decomposition **exact**. See the embedding ablation in Supplementary Table S2.
- **Kernel PCA**: nonlinear and uncorrelated, but latents are typically
  non-Gaussian; requires arbitrary polynomial chaos (aPC) for exact Sobol.
- **PCA + Normalizing Flow**: achieves exact Gaussianity but can introduce
  residual correlations at small sample sizes.

For this demo we use PCA for transparency; see `core/embeddings.py` for a
reusable β-VAE embedding that can serve as a drop-in replacement.

## 3. Setup and imports

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

# Add project root to path
ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(ROOT))

from sklearn.decomposition import PCA
from core.pce_surrogate import (
    PCESurrogate, TrajectoryPCESurrogate,
    hermite_basis, build_design_matrix, alr_transform, alr_inverse,
    calibration_coverage, run_ks_battery,
)
from core.distributional_analysis import (
    detect_bimodality, analyse_trajectory_bimodality,
    plot_bimodality_panel, plot_surrogate_comparison,
)
from experiments.gridworld.run_experiment import (
    GridGFlowNet, sample_reward_configs, train_single_member,
    compute_reward_grid, GRID_SIZE, DISCRETE_ACTIONS, DISCRETE_N_STEPS,
    ZONE_MAP,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print(f'Project root: {ROOT}')
print(f'Grid size: {GRID_SIZE}x{GRID_SIZE}, Actions: {DISCRETE_ACTIONS}, Steps: {DISCRETE_N_STEPS}')

## 4. The gridworld environment

Our agent navigates a 5x5 grid. The reward at each cell is determined by
4 **zone shift** parameters -- one per quadrant. Different zone shifts
create different reward landscapes, and the agent's policy changes accordingly.

The **uncertainty question**: if we don't know the zone shifts precisely,
how does that propagate to the agent's decisions?

In [ ]:
# Visualise the zone map and a few example reward grids
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

# Zone map
im = axes[0].imshow(ZONE_MAP, cmap='Set3', interpolation='nearest')
axes[0].set_title('Zone map (4 quadrants)', fontweight='bold')
for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        axes[0].text(c, r, str(ZONE_MAP[r, c]), ha='center', va='center', fontsize=9)

# Three example reward grids with different zone shifts
examples = [
    ([1.0, -1.0, 0.0, 0.5], 'Shift: [1, -1, 0, 0.5]'),
    ([-0.5, 0.5, 1.0, -1.0], 'Shift: [-0.5, 0.5, 1, -1]'),
    ([0.0, 0.0, 1.0, 1.0], 'Shift: [0, 0, 1, 1]'),
]
for i, (shifts, title) in enumerate(examples):
    grid = compute_reward_grid(np.array(shifts))
    im = axes[i+1].imshow(grid, cmap='RdYlGn', interpolation='nearest')
    axes[i+1].set_title(title, fontsize=9)
    plt.colorbar(im, ax=axes[i+1], shrink=0.8)

plt.suptitle('Reward landscapes depend on uncertain zone parameters', fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 5. Train an ensemble of GFlowNets

The core idea: **different plausible reward functions lead to different
trained policies**. We sample many reward configurations, train a GFlowNet
on each, and collect the resulting policy distributions.

This ensemble is the raw material for the PCE surrogate.

In [ ]:
# Parameters (small for demo speed; paper uses larger ensembles)
N_TRAIN = 25
N_TEST  = 40
N_TOTAL = N_TRAIN + N_TEST
N_EPISODES = 200       # training episodes per GFlowNet
PCE_DEGREE = 3
PCA_DIM = 2

print(f'Training {N_TOTAL} GFlowNets ({N_EPISODES} episodes each)...')
print(f'This takes ~{N_TOTAL * 3}s on a modern CPU.')

# Sample reward configurations
params, grids = sample_reward_configs(N_TOTAL, 'discrete', seed=42)

# Train ensemble
all_policies = []
for i in range(N_TOTAL):
    pol, _ = train_single_member(grids[i], mode='discrete', seed=i, n_episodes=N_EPISODES)
    all_policies.append(pol)
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_TOTAL} done')

# Organise into per-step arrays
n_steps = DISCRETE_N_STEPS
tr_pol = {s: np.array([all_policies[i][s] for i in range(N_TRAIN)]) for s in range(n_steps)}
te_pol = {s: np.array([all_policies[N_TRAIN + i][s] for i in range(N_TEST)]) for s in range(n_steps)}

print(f'\nPolicy shape per step: {tr_pol[0].shape}  (n_members, n_actions)')
print('Done.')

## 6. Dimensionality reduction: PCA on reward configurations

The reward is parameterised by 25 cell values (5x5 grid). We use PCA to
reduce this to 2 principal components $\mu_1, \mu_2$.

**Why PCA?** Because PCA components are uncorrelated (and approximately
independent under Gaussianity), which is exactly what the Sobol formula
requires. Each $\mu_i$ is a "knob" that independently controls one aspect
of the reward landscape.

In [ ]:
# PCA on flattened reward grids
flat_grids = grids.reshape(N_TOTAL, -1)
pca = PCA(n_components=PCA_DIM)
mu = pca.fit_transform(flat_grids)
mu_tr, mu_te = mu[:N_TRAIN], mu[N_TRAIN:]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Scatter plot of PCA coordinates
axes[0].scatter(mu_tr[:, 0], mu_tr[:, 1], c='steelblue', alpha=0.7, label='Train', edgecolors='white', s=50)
axes[0].scatter(mu_te[:, 0], mu_te[:, 1], c='coral', alpha=0.7, label='Test', edgecolors='white', s=50)
axes[0].set_xlabel('$\\mu_1$ (PC1)')
axes[0].set_ylabel('$\\mu_2$ (PC2)')
axes[0].set_title('Reward parameterisation\n(PCA of 25D grid -> 2D)', fontweight='bold')
axes[0].legend()

# Explained variance
axes[1].bar([1, 2], pca.explained_variance_ratio_ * 100, color=['steelblue', 'coral'])
axes[1].set_xlabel('Principal component')
axes[1].set_ylabel('Explained variance (%)')
axes[1].set_title(f'PCA: {pca.explained_variance_ratio_.sum()*100:.1f}% total', fontweight='bold')

# Visualise what the PCs correspond to
for idx, (ax_i, label) in enumerate([(axes[2], 'PC1'), (axes[2], 'PC2')]):
    pass
# Show PC1 loading reshaped as grid
pc1_grid = pca.components_[0].reshape(GRID_SIZE, GRID_SIZE)
im = axes[2].imshow(pc1_grid, cmap='RdBu_r', interpolation='nearest')
axes[2].set_title('PC1 loading pattern\n(what $\\mu_1$ controls)', fontweight='bold')
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, '
      f'PC2={pca.explained_variance_ratio_[1]:.3f}, '
      f'Total={pca.explained_variance_ratio_.sum():.3f}')

## 7. The Hermite polynomial basis -- visualised

Before fitting the PCE, let's see what the basis functions look like.
These are the building blocks of our surrogate: orthonormal polynomials
weighted by the Gaussian density.

In [ ]:
x = np.linspace(-3, 3, 200)
basis = hermite_basis(x, degree=5)
gaussian = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot basis functions
colors = plt.cm.viridis(np.linspace(0.1, 0.9, 6))
for n in range(6):
    axes[0].plot(x, basis[n], label=f'$\\psi_{n}$', color=colors[n], linewidth=1.5)
axes[0].set_xlabel('$\\mu$')
axes[0].set_ylabel('$\\psi_n(\\mu)$')
axes[0].set_title('Hermite basis functions (degree 0-5)', fontweight='bold')
axes[0].legend(ncol=2, fontsize=8)
axes[0].set_ylim(-3, 3)
axes[0].axhline(0, color='gray', linewidth=0.5)

# Plot weighted basis (what the inner product "sees")
for n in range(6):
    axes[1].plot(x, basis[n] * gaussian, color=colors[n], linewidth=1.5, label=f'$\\psi_{n} \\cdot \\phi$')
axes[1].fill_between(x, gaussian, alpha=0.15, color='gray', label='$\\mathcal{N}(0,1)$')
axes[1].set_xlabel('$\\mu$')
axes[1].set_title('Basis $\\times$ Gaussian density\n(orthogonality is w.r.t. this weight)', fontweight='bold')
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

# Verify orthonormality numerically
dx = x[1] - x[0]
G = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        G[i, j] = np.sum(basis[i] * basis[j] * gaussian) * dx
print('Gram matrix (should be ~identity):')
print(np.round(G, 3))

## 8. Fitting the PCE surrogate

Now we fit a PCE that maps $\mu = (\mu_1, \mu_2) \to \pi(a|s)$ at each
trajectory step. The fitting involves:

1. **ALR transform**: map simplex probabilities to unconstrained log-ratios
2. **Build design matrix**: evaluate all multivariate Hermite basis functions at training $\mu$
3. **Ridge regression**: find coefficients $c_j$ that best fit the ALR values
4. **Sobol indices**: read off sensitivity from the squared coefficients

In [ ]:
# Fit the trajectory PCE surrogate
tsurr = TrajectoryPCESurrogate(degree=PCE_DEGREE, basis='hermite')
for s in range(n_steps):
    tsurr.fit_step(s, mu_tr, tr_pol[s])

# How many PCE terms?
n_terms = tsurr.step_surrogates[0].multi_idx.shape[0]
n_alr = DISCRETE_ACTIONS - 1
print(f'PCE degree: {PCE_DEGREE}')
print(f'Input dimension: {PCA_DIM}')
print(f'Number of basis terms: {n_terms}')
print(f'ALR components per step: {n_alr}')
print(f'Total coefficients per step: {n_terms} x {n_alr} = {n_terms * n_alr}')
print(f'Training samples: {N_TRAIN}')
print(f'  -> ratio L/P = {N_TRAIN / n_terms:.1f} (>1 required for fitting)')

# Show the multi-index structure
mi = tsurr.step_surrogates[0].multi_idx
print(f'\nMulti-index array (first 10 of {len(mi)}):')
print('  j = (j1, j2)  |  meaning')
print('  ' + '-' * 40)
for idx in range(min(10, len(mi))):
    j = mi[idx]
    if j.sum() == 0:
        meaning = 'constant (mean)'
    elif j[1] == 0:
        meaning = f'pure mu_1 effect (degree {j[0]})'
    elif j[0] == 0:
        meaning = f'pure mu_2 effect (degree {j[1]})'
    else:
        meaning = f'interaction mu_1^{j[0]} * mu_2^{j[1]}'
    print(f'  {tuple(j)}       |  {meaning}')

## 9. Sobol sensitivity indices

Now the payoff: we read off Sobol indices directly from the PCE coefficients.

- $S_1$ (first-order, PC1): fraction of policy variance due to $\mu_1$ alone
- $S_2$ (first-order, PC2): fraction due to $\mu_2$ alone
- $T_1, T_2$ (total-order): includes interaction effects

This tells us **which aspect of reward uncertainty drives which decisions**.

In [ ]:
sobol = tsurr.sobol_all_steps()

# Print the accessible narrative
print(tsurr.summarise_sobol(
    step_labels={s: f'Move {s}' for s in range(n_steps)},
    dim_labels=['PC1 (zone-level shifts)', 'PC2 (zone contrast)'],
))

In [ ]:
# Visualise Sobol indices as a stacked bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps_arr = np.arange(n_steps)
width = 0.35

for plot_idx, (order_key, title) in enumerate([
    ('first_order', 'First-order Sobol indices'),
    ('total_order', 'Total-order Sobol indices'),
]):
    ax = axes[plot_idx]
    # Average over ALR components (actions) for the headline
    s1_vals = [sobol[s][order_key][:, 0].mean() for s in range(n_steps)]
    s2_vals = [sobol[s][order_key][:, 1].mean() for s in range(n_steps)]
    
    bars1 = ax.bar(steps_arr - width/2, s1_vals, width, label='$\\mu_1$ (PC1)',
                   color='steelblue', edgecolor='white')
    bars2 = ax.bar(steps_arr + width/2, s2_vals, width, label='$\\mu_2$ (PC2)',
                   color='coral', edgecolor='white')
    ax.set_xlabel('Trajectory step')
    ax.set_ylabel('Sobol index')
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(steps_arr)
    ax.set_xticklabels([f'Move {s}' for s in range(n_steps)])
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)

plt.suptitle('Which reward component drives which decision?', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. How the Sobol formula works: a coefficient-level view

Let's look inside the PCE for one step and see exactly how
the squared coefficients partition the variance.

In [ ]:
# Pick the step with highest variance for illustration
step_variances = [sobol[s]['variance'].max() for s in range(n_steps)]
demo_step = int(np.argmax(step_variances))
demo_surr = tsurr.step_surrogates[demo_step]
demo_alr = int(np.argmax(sobol[demo_step]['variance']))  # highest-variance ALR component

c = demo_surr.coefficients[demo_alr]
mi = demo_surr.multi_idx
c2 = c**2

# Classify each term
colors_terms = []
labels_terms = []
for idx, j in enumerate(mi):
    if idx == 0:
        colors_terms.append('lightgray')   # constant
        labels_terms.append('const')
    elif j[1] == 0:
        colors_terms.append('steelblue')   # pure PC1
        labels_terms.append('PC1')
    elif j[0] == 0:
        colors_terms.append('coral')       # pure PC2
        labels_terms.append('PC2')
    else:
        colors_terms.append('mediumpurple') # interaction
        labels_terms.append('PC1xPC2')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of c_j^2
axes[0].bar(range(len(c2)), c2, color=colors_terms, edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Basis term index $j$')
axes[0].set_ylabel('$c_j^2$ (variance contribution)')
axes[0].set_title(f'Step {demo_step}, ALR component {demo_alr}:\n'
                  f'squared coefficients partition the variance', fontweight='bold')

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', label='Constant (mean)'),
    Patch(facecolor='steelblue', label='Pure $\\mu_1$ terms -> $S_1$'),
    Patch(facecolor='coral', label='Pure $\\mu_2$ terms -> $S_2$'),
    Patch(facecolor='mediumpurple', label='Interaction terms'),
]
axes[0].legend(handles=legend_elements, fontsize=8)

# Pie chart of variance decomposition
D_total = c2[1:].sum()
pc1_var = sum(c2[i] for i in range(len(mi)) if i > 0 and mi[i][1] == 0)
pc2_var = sum(c2[i] for i in range(len(mi)) if i > 0 and mi[i][0] == 0)
inter_var = D_total - pc1_var - pc2_var

sizes = [pc1_var, pc2_var, inter_var]
pie_labels = [f'$\\mu_1$ only\n({pc1_var/D_total:.1%})',
              f'$\\mu_2$ only\n({pc2_var/D_total:.1%})',
              f'Interaction\n({inter_var/D_total:.1%})']
pie_colors = ['steelblue', 'coral', 'mediumpurple']
axes[1].pie(sizes, labels=pie_labels, colors=pie_colors, autopct='',
            startangle=90, textprops={'fontsize': 9})
axes[1].set_title(f'Variance decomposition\n(total D = {D_total:.4f})', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'S_1 (first-order) = {sobol[demo_step]["first_order"][demo_alr, 0]:.3f}')
print(f'S_2 (first-order) = {sobol[demo_step]["first_order"][demo_alr, 1]:.3f}')
print(f'T_1 (total-order) = {sobol[demo_step]["total_order"][demo_alr, 0]:.3f}')
print(f'T_2 (total-order) = {sobol[demo_step]["total_order"][demo_alr, 1]:.3f}')

## 11. Surrogate validation: does the PCE match reality?

A surrogate is only useful if it faithfully represents the true distribution.
We validate the PCE against the held-out test ensemble using:

- **KS test**: does each action's marginal distribution match?
- **Calibration coverage**: do 90% credible intervals actually contain 90% of test points?

In [ ]:
# KS test battery
ks = run_ks_battery(tsurr, te_pol)
print('KS test results (H0: surrogate and empirical are from same distribution):')
for s in range(n_steps):
    fr = ks[s]['fraction_pass']
    print(f'  Step {s}: {ks[s]["n_pass"]}/{ks[s]["n_total"]} actions pass ({fr:.0%})')

# Calibration
mc_pols = tsurr.sample_trajectory_policies(5000)
print('\nCalibration coverage (nominal -> empirical):')
for s in range(n_steps):
    cov = calibration_coverage(mc_pols[s], te_pol[s])
    parts = [f'{100*lv:.0f}%->{cov[lv]:.2f}' for lv in sorted(cov)]
    print(f'  Step {s}: {", ".join(parts)}')

In [ ]:
# Visual comparison: surrogate vs empirical
fig = plot_surrogate_comparison(
    mc_pols, te_pol, action_idx=0,
    step_labels={s: f'Move {s}' for s in range(n_steps)},
    action_name='up',
    title='PCE surrogate vs empirical: P(up) at each step',
)
plt.show()

## 12. Beyond Sobol: detecting behavioural bifurcations

Sobol indices tell us *how much* variance comes from each reward component,
but nothing about the *shape* of that uncertainty. The PCE surrogate is a
full generative model -- we can sample from it and inspect the resulting
distributions directly.

**The key insight**: at some trajectory steps, the stop-action probability
may be **bimodal**. This means that under some reward configurations the
GFlowNet terminates confidently, while under others it continues exploring.
These are qualitatively different behavioural regimes that a scalar
variance measure cannot distinguish.

This is directly **actionable**: it identifies the parameter region where
additional data would most reduce structural ambiguity.

In [ ]:
# Bimodality analysis on the stop action
stop_action = DISCRETE_ACTIONS - 1  # last action = stop
step_labels = {s: f'Move {s}' for s in range(n_steps)}

bimodality, narrative = analyse_trajectory_bimodality(
    mc_pols, action_idx=stop_action, step_labels=step_labels,
)
print(narrative)

In [ ]:
# Publication-quality bimodality panel
fig = plot_bimodality_panel(
    mc_pols, action_idx=stop_action,
    step_labels=step_labels,
    action_name='stop',
    title='Stop-action distributional structure across trajectory',
    empirical_samples=te_pol,
)
plt.show()

## 13. The PCE surface: policy as a function of reward parameters

Because $\mu$ is 2D, we can directly visualise how the policy changes
as we move through the reward parameter space. This is the "response surface"
that the PCE surrogate approximates.

In [ ]:
# Evaluate the PCE on a dense grid in (mu_1, mu_2) space
grid_n = 50
mu1_range = np.linspace(mu_tr[:, 0].min() - 0.5, mu_tr[:, 0].max() + 0.5, grid_n)
mu2_range = np.linspace(mu_tr[:, 1].min() - 0.5, mu_tr[:, 1].max() + 0.5, grid_n)
M1, M2 = np.meshgrid(mu1_range, mu2_range)
mu_grid = np.column_stack([M1.ravel(), M2.ravel()])

demo_surr_obj = tsurr.step_surrogates[demo_step]
pred_pol = demo_surr_obj.predict(mu_grid)  # (grid_n^2, K)

fig, axes = plt.subplots(1, DISCRETE_ACTIONS, figsize=(16, 3.5))
action_names = ['Up', 'Down', 'Left', 'Right', 'Stop']

for a in range(DISCRETE_ACTIONS):
    Z = pred_pol[:, a].reshape(grid_n, grid_n)
    im = axes[a].contourf(M1, M2, Z, levels=20, cmap='viridis')
    axes[a].scatter(mu_tr[:, 0], mu_tr[:, 1], c='white', s=8, alpha=0.5, edgecolors='none')
    axes[a].set_xlabel('$\\mu_1$')
    if a == 0:
        axes[a].set_ylabel('$\\mu_2$')
    axes[a].set_title(f'P({action_names[a]})', fontweight='bold')
    plt.colorbar(im, ax=axes[a], shrink=0.8)

plt.suptitle(f'PCE response surface at step {demo_step}: policy as a function of $(\\mu_1, \\mu_2)$',
             fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 14. Sample complexity: how many GFlowNets do we need?

The PCE framework comes with a **theoretical bound** (Theorem A in the paper)
on the minimum ensemble size needed for a target Sobol accuracy.

In [ ]:
L_required = tsurr.required_ensemble_size(target_sobol_error=0.05, confidence=0.95)
print(f'Theoretical minimum ensemble size for |S_hat - S| < 0.05 (95% confidence): {L_required}')
print(f'We used: {N_TRAIN} training members')
print(f'PCE terms: {n_terms}')
print(f'\nNote: the bound is conservative; in practice the PCE converges faster')
print(f'because the polynomial structure regularises the fit.')

## 15. Summary: why PCE + GFlowNet?

### The algebraic fit

GFlowNets and PCEs are a natural pairing:

| GFlowNet property | PCE capability | Result |
|-------------------|---------------|--------|
| Policy lives on the **simplex** | ALR transform maps to $\mathbb{R}^{K-1}$ | PCE operates in unconstrained space |
| Reward is **uncertain** | PCE captures input-output mapping | Full policy *distribution*, not just one policy |
| Decisions are **sequential** | Per-step PCE | Sensitivity indices *per decision step* |
| **Orthonormal** Hermite basis | PCA gives independent Gaussian inputs | Sobol indices **analytically** from coefficients |

### What the framework delivers

1. **Sobol indices**: which reward components matter for which decisions (closed-form)
2. **Distributional structure**: bimodality reveals qualitative behavioural regimes
3. **Calibrated uncertainty**: credible intervals that match empirical coverage
4. **Sample complexity bound**: minimum ensemble size for target accuracy
5. **Actionable insights**: directly identifies where additional data would most reduce ambiguity

### Embedding alternatives: what our ablation showed

The default PCA embedding is simple and effective, but nonlinear alternatives
can strengthen the theoretical guarantees:

| Embedding | Variance / $R^2$ | Independent | Gaussian | Sobol validity |
|-----------|:---:|:---:|:---:|:---|
| **Linear PCA** | 0.45 | Yes | No | Approx. (indep., non-Gaussian) |
| **Kernel PCA** | — | Yes | No | Approx. (indep., non-Gaussian) |
| **β-VAE (β=4)** | **0.996** | Yes | **Yes** | **Exact** (indep. + Gaussian) |
| **PCA + NF** | 0.45 | No | Yes | Caution (Gaussian, correlated) |

The β-VAE achieves near-perfect reconstruction while producing latents that
are both approximately independent and Gaussian -- making the analytical
Sobol decomposition **exact** rather than approximate. Crucially, **calibration
remains excellent across all methods** (≥ 0.98), confirming that PCA captures
the policy-relevant variation even when total variance explained is modest.

See `core/embeddings.py` for a drop-in β-VAE embedding, and Supplementary
Table S2 for full results on the Buchwald-Hartwig benchmark.

### What MC Sobol misses

- **MC Sobol (Saltelli)**: requires $(d+2) \times N$ model evaluations; PCE needs none after fitting
- **Bootstrap**: captures uncertainty *in* the index but not the decomposition structure

The PCE approach is simultaneously **cheaper** (no MC), **richer** (full distribution,
not just variance), and **more interpretable** (coefficient-level decomposition).